# 18. Monotonic Stack and Monotonic Queue

**Topics**: Stack, Queue, Sliding Window, Array Patterns, Optimization

---

## Overview

This comprehensive chapter covers:

- Theory and intuition behind monotonic structures
- Monotonic stack for next greater/smaller problems
- Monotonic queue for sliding window max/min
- Complexity analysis and amortized O(n) reasoning
- Pattern recognition and when to use each technique
- Complete LeetCode problem implementations
- Common mistakes and best practices

---

## 1. The Core Idea: A Real-World Analogy

### Imagine You're in a Line of People

You're standing in a line with people of different heights. For each person, you want to know: **"Who is the next taller person behind me?"**

**Naive approach**: Each person turns around and looks at everyone behind them one by one until they find someone taller. If there are 100 people, person #1 might check 99 people. Person #2 checks 98 people. This is **O(n²)** - very slow!

**Monotonic stack approach**: As people join the line, we keep a special list of "people still waiting for their answer." When a tall person joins:
- They become the answer for everyone shorter in front of them
- Those shorter people can leave the waiting list (they got their answer!)
- The tall person joins the waiting list

This way, each person is added to the waiting list once and removed once. **O(n)** - much faster!

### The Problem

Many array problems ask:

- **Next Greater Element**: Find the next element to the right that is larger
- **Next Smaller Element**: Find the next element to the right that is smaller
- **Sliding Window Maximum**: Find max in every window of size k
- **Rectangle Problems**: Find areas based on height boundaries

### Why Brute Force Fails: O(n²)

```python
# For each element, scan everything to the right
for i in range(n):
    for j in range(i+1, n):
        if nums[j] > nums[i]:  # Found it!
            answer[i] = nums[j]
            break
```

**Problem**: We repeat comparisons. Element at index 0 might compare with all n-1 elements. Element at index 1 compares with n-2 elements. Total: n² comparisons.

### Monotonic Solution: O(n)

**Key insight**: If we know element A will never be useful again, remove it immediately.

**Example**: Array `[5, 3, 7]`
- When we see 7, we know 5 and 3 both have 7 as their answer
- We also know 5 and 3 can NEVER be the answer for any future element (7 is bigger and came later)
- So we remove 5 and 3 from consideration forever

By removing useless candidates early, we reduce to **O(n)**.

---

## 2. Understanding "Monotonic" - What Does It Really Mean?

### Simple Definition

**Monotonic** = moving in one consistent direction (always up or always down)

Think of it like:
- **Climbing stairs**: Each step goes up (or stays level) - monotonically increasing
- **Going downhill**: Each step goes down (or stays level) - monotonically decreasing

### Examples

**Monotonically increasing**: `[1, 2, 2, 5, 8, 10]`
```
10 |                    *
 8 |                *
 5 |            *
 2 |    *   *
 1 | *
```
Each value ≥ previous value (can be equal)

**Monotonically decreasing**: `[10, 9, 7, 7, 3, 1]`
```
10 | *
 9 |    *
 7 |        *   *
 3 |                *
 1 |                    *
```
Each value ≤ previous value (can be equal)

**Strictly increasing**: `[1, 2, 5, 8, 10]` (no equals allowed)

**Strictly decreasing**: `[10, 9, 7, 3, 1]` (no equals allowed)

### Critical Understanding

**The original array does NOT need to be monotonic!**

For example, `[2, 1, 2, 4, 3]` is NOT monotonic (it goes down, then up, then down).

**But our stack/queue will maintain monotonic order.**

We keep our data structure organized so values inside it are always in increasing or decreasing order.

### Why This Helps

If our stack is decreasing (like `[10, 7, 3]`), and we see a new value `5`:
- We know `5 > 3`, so 3 got its answer
- We know `5 < 7`, so 7 is still waiting
- The stack becomes `[10, 7, 5]` - still decreasing!

This organization lets us quickly find answers without checking every element.

---

## 3. Monotonic Stack: The Complete Intuition

### What Is a Monotonic Stack?

A **monotonic stack** is a regular stack, but we enforce a rule: **values must stay in increasing or decreasing order**.

When a new element breaks the order, we pop elements until the order is restored.

### Two Types Explained Simply

#### Type 1: Monotonic Decreasing Stack (for Next Greater Element)

**Rule**: Stack values decrease from bottom to top

**Visual**:
```
Stack (bottom to top): [10, 7, 5, 3]
         Bottom → Top
         Larger → Smaller
```

**When to use**: Finding the **next greater element**

**Why it works**: 
- When we see a large number (say 8), it's greater than 3, 5, and 7
- We pop 3, 5, 7 and record "8 is their answer"
- Stack becomes [10, 8] - still decreasing!

#### Type 2: Monotonic Increasing Stack (for Next Smaller Element)

**Rule**: Stack values increase from bottom to top

**Visual**:
```
Stack (bottom to top): [1, 3, 5, 7]
         Bottom → Top
         Smaller → Larger
```

**When to use**: Finding the **next smaller element**

**Why it works**:
- When we see a small number (say 4), it's smaller than 7 and 5
- We pop 7 and 5 and record "4 is their answer"
- Stack becomes [1, 3, 4] - still increasing!

### Why Store Indices Instead of Values?

**Critical**: We almost always store **indices**, not the actual values.

**Why?**

1. **We can get the value anytime**: `nums[index]` gives us the value
2. **We need distances**: "How many days until warmer?" = `current_index - previous_index`
3. **We need positions**: "Which elements are in the current window?"
4. **We need ranges**: "How far can this bar extend?"

**Example**:
```python
nums = [2, 1, 2, 4, 3]
stack = [0, 1, 2]  # indices

# We can access values:
nums[stack[-1]]  # nums[2] = 2

# We can compute distance:
i - stack[-1]  # distance between current and top
```

### The Core Invariant (The Rule That Never Breaks)

For a **decreasing stack** used for next greater element:

```
If index i is below index j in the stack,
then nums[i] > nums[j]

Example stack: [0, 2, 4]
If nums = [5, 1, 3, 1, 2]
Then: nums[0]=5 > nums[2]=3 > nums[4]=2 ✓
```

**What this means**: 
- The bottom of the stack has the largest value
- The top of the stack has the smallest value
- Everything is in decreasing order

This organization is what makes the algorithm fast!

---

## 4. Next Greater Element: Step-by-Step Intuition

### The Problem in Plain English

Given an array, for each element, find the **first element to its right** that is **strictly larger**.

### Example Walkthrough

```
Input:  [2, 1, 2, 4, 3]
Output: [4, 2, 4, -1, -1]

Why?
- 2: next greater is 4 (skip 1 and 2, they're not greater)
- 1: next greater is 2 (the one at index 2)
- 2: next greater is 4
- 4: no greater element exists → -1
- 3: no greater element exists → -1
```

### Brute Force vs Monotonic Stack

**Brute Force** (what NOT to do):
```python
for i in range(n):
    for j in range(i+1, n):  # Check everything to the right
        if nums[j] > nums[i]:
            answer[i] = nums[j]
            break
```
**Time**: O(n²) - for each element, scan all elements to the right

**Monotonic Stack** (the smart way):
```python
stack = []  # indices waiting for their answer
for i, num in enumerate(nums):
    # Current num is the answer for all smaller elements in stack
    while stack and nums[stack[-1]] < num:
        answer[stack.pop()] = num
    stack.append(i)  # Current element is now waiting
```
**Time**: O(n) - each element pushed once, popped once

### Why the Monotonic Stack Works

**Key insight**: When we see a large number, it immediately answers all smaller numbers waiting in the stack.

**Visual Example**: `[2, 1, 2, 4, 3]`

```
Step 0: See 2
  Stack: [0]  (index 0, value 2)
  Waiting: [2]

Step 1: See 1
  1 < 2, so 2 is still waiting
  Stack: [0, 1]  (indices 0,1, values 2,1)
  Waiting: [2, 1]

Step 2: See 2
  2 > 1! So 1 found its answer: 2
  Pop index 1, set answer[1] = 2
  2 = 2, so index 0 keeps waiting
  Stack: [0, 2]  (indices 0,2, values 2,2)
  Waiting: [2, 2]

Step 3: See 4
  4 > 2! Both waiting 2's found their answer: 4
  Pop index 2, set answer[2] = 4
  Pop index 0, set answer[0] = 4
  Stack: [3]  (index 3, value 4)
  Waiting: [4]

Step 4: See 3
  3 < 4, so 4 is still waiting
  Stack: [3, 4]  (indices 3,4, values 4,3)
  Waiting: [4, 3]

Done! Elements still in stack have no answer → -1
```

### The "Aha!" Moment

**Why can we remove elements from the stack?**

When we see 4 at index 3, and pop 2 at index 2:
- We found that 4 is the answer for 2
- Can 2 ever be the answer for anyone else? **NO!**
  - Any future element asking "what's my next greater?" will see 4 before seeing 2
  - 4 is bigger than 2 AND comes later
  - So 2 is now useless, we can forget about it

**This is why it's O(n)**: Each element gets pushed once and popped at most once. No element is checked multiple times.

In [ ]:
def next_greater_element(nums):
    """
    Find next greater element for each position.
    Time: O(n), Space: O(n)
    """
    result = [-1] * len(nums)
    stack = []  # store indices

    for i, num in enumerate(nums):
        # Pop all smaller elements - current is their answer
        while stack and nums[stack[-1]] < num:
            prev_index = stack.pop()
            result[prev_index] = num
        
        stack.append(i)

    return result

# Test
test_cases = [
    [2, 1, 2, 4, 3],
    [1, 2, 3, 4, 5],
    [5, 4, 3, 2, 1],
]

for nums in test_cases:
    result = next_greater_element(nums)
    print(f"nums = {nums}")
    print(f"next greater = {result}\n")

In [ ]:
def next_greater_element_verbose(nums):
    """Verbose version showing each step."""
    result = [-1] * len(nums)
    stack = []

    print(f"Processing: {nums}\n")

    for i, num in enumerate(nums):
        print(f"Step {i}: value = {num}")
        print(f"  Stack before: {stack} -> {[nums[idx] for idx in stack]}")

        popped = []
        while stack and nums[stack[-1]] < num:
            prev = stack.pop()
            result[prev] = num
            popped.append(prev)
        
        if popped:
            print(f"  Popped {popped}, set answers to {num}")
        
        stack.append(i)
        print(f"  Stack after: {stack}")
        print(f"  Result: {result}\n")

    return result

next_greater_element_verbose([2, 1, 2, 4, 3])

---

## 5. Next Smaller Element

In [ ]:
def next_smaller_element(nums):
    """
    Find next smaller element for each position.
    Time: O(n), Space: O(n)
    """
    result = [-1] * len(nums)
    stack = []

    for i, num in enumerate(nums):
        while stack and nums[stack[-1]] > num:
            prev = stack.pop()
            result[prev] = num
        stack.append(i)

    return result

nums = [5, 2, 8, 6, 3]
print(f"nums = {nums}")
print(f"next smaller = {next_smaller_element(nums)}")

---

## 6. Previous Greater and Smaller

In [ ]:
def previous_greater_element(nums):
    result = [-1] * len(nums)
    stack = []

    for i, num in enumerate(nums):
        while stack and nums[stack[-1]] <= num:
            stack.pop()
        if stack:
            result[i] = nums[stack[-1]]
        stack.append(i)

    return result

def previous_smaller_element(nums):
    result = [-1] * len(nums)
    stack = []

    for i, num in enumerate(nums):
        while stack and nums[stack[-1]] >= num:
            stack.pop()
        if stack:
            result[i] = nums[stack[-1]]
        stack.append(i)

    return result

nums = [3, 7, 1, 7, 8, 4]
print(f"nums = {nums}")
print(f"previous greater = {previous_greater_element(nums)}")
print(f"previous smaller = {previous_smaller_element(nums)}")

---

## 7. Why Monotonic Stack Is O(n) - The Math Explained

### The Concern

Looking at the code, you see a `while` loop inside a `for` loop:

```python
for i in range(n):           # n iterations
    while stack and ...:     # ??? iterations
        stack.pop()
```

This looks like O(n²), right? **Wrong!**

### The Reality: Amortized Analysis

**Key observation**: Each element can only be popped once because once it's popped, it's gone forever.

Let's count operations for `[2, 1, 2, 4, 3]`:

```
Element | Push | Pop | Total Ops
--------|------|-----|----------
   2    |  1   |  1  |    2
   1    |  1   |  1  |    2
   2    |  1   |  1  |    2
   4    |  1   |  0  |    1
   3    |  1   |  0  |    1
--------|------|-----|----------
Total   |  5   |  3  |    8
```

**5 pushes + 3 pops = 8 operations for 5 elements**

### General Formula

For an array of size n:
- **Maximum pushes**: n (one per element)
- **Maximum pops**: n (can't pop more than we pushed)
- **Total operations**: at most 2n

**Time complexity**: O(2n) = **O(n)**

### Visual Proof

Think of it like a turnstile at a stadium:
- Each person enters once (push)
- Each person exits at most once (pop)
- If 1000 people attend, there are at most 2000 turnstile operations

No person goes through the turnstile 1000 times!

### Why "Amortized"?

Some iterations of the for loop do a lot of work (pop many elements), others do little (pop none).

But **on average**, each iteration does constant work because the total pops across all iterations is bounded by n.

**This is called amortized O(n)** - the cost is "spread out" over all operations.

---

## 8. Monotonic Queue: Sliding Window Maximum Explained

### The Problem

Given an array and a window size k, find the maximum value in each window as it slides.

**Example**:
```
nums = [1, 3, -1, -3, 5, 3, 6, 7], k = 3

Windows:
[1  3  -1] -3  5  3  6  7  → max = 3
 1 [3  -1  -3] 5  3  6  7  → max = 3
 1  3 [-1  -3  5] 3  6  7  → max = 5
 1  3  -1 [-3  5  3] 6  7  → max = 5
 1  3  -1  -3 [5  3  6] 7  → max = 6
 1  3  -1  -3  5 [3  6  7] → max = 7

Output: [3, 3, 5, 5, 6, 7]
```

### Naive Approach: O(nk)

```python
for i in range(n - k + 1):
    window_max = max(nums[i:i+k])  # Check all k elements
    result.append(window_max)
```

For each window, scan all k elements. Total: O(nk)

### Monotonic Queue Approach: O(n)

**Key insight**: We don't need to keep ALL elements in the window, only the ones that could potentially be the maximum.

**When is an element useless?**

If we have two elements in the window:
- Element A with value 3 at index 2
- Element B with value 5 at index 4

Element A is **useless** because:
1. B is larger (5 > 3)
2. B will stay in the window longer (it came later)
3. So A can never be the maximum while B is around

**We remove A immediately!**

### How the Deque Works

We use a **deque** (double-ended queue) that stores indices in **decreasing order of values**.

**Two operations**:
1. **Remove from front**: When indices fall outside the window (expired)
2. **Remove from back**: When new element makes old elements useless (too small)

**Visual Example**: `[1, 3, -1, -3, 5, 3, 6, 7]`, k=3

```
Step 0: See 1
  Deque: [0]  (value 1)
  
Step 1: See 3
  3 > 1, so 1 is useless, remove it
  Deque: [1]  (value 3)
  
Step 2: See -1
  -1 < 3, so 3 might still be max
  Deque: [1, 2]  (values 3, -1)
  Window [1,3,-1]: max = 3 (front of deque)
  
Step 3: See -3
  -3 < -1, keep both
  Deque: [1, 2, 3]  (values 3, -1, -3)
  Window [3,-1,-3]: max = 3
  
Step 4: See 5
  Index 1 is now outside window (i=4, k=3, so i-k=1)
  Remove index 1 from front
  5 > -1 and -3, remove both from back
  Deque: [4]  (value 5)
  Window [-1,-3,5]: max = 5
  
Step 5: See 3
  3 < 5, keep 5
  Deque: [4, 5]  (values 5, 3)
  Window [-3,5,3]: max = 5
  
Step 6: See 6
  Index 4 still in window
  6 > 3, remove 3
  6 > 5, remove 5
  Deque: [6]  (value 6)
  Window [5,3,6]: max = 6
  
Step 7: See 7
  7 > 6, remove 6
  Deque: [7]  (value 7)
  Window [3,6,7]: max = 7
```

### Why Deque Front Is Always the Maximum

The deque maintains values in **decreasing order**:
- Front has the largest value
- Back has the smallest value

When we add a new element:
- Remove smaller elements from back (they're useless)
- The front is still the largest

When the window slides:
- Remove expired elements from front
- The new front is the next largest

**The front always points to the maximum in the current window!**

In [ ]:
from collections import deque

def sliding_window_max(nums, k):
    """
    Find maximum in each sliding window of size k.
    Time: O(n), Space: O(k)
    """
    q = deque()
    result = []

    for i, num in enumerate(nums):
        # Remove expired indices
        while q and q[0] <= i - k:
            q.popleft()

        # Remove smaller values
        while q and nums[q[-1]] <= num:
            q.pop()

        q.append(i)

        if i >= k - 1:
            result.append(nums[q[0]])

    return result

nums = [1, 3, -1, -3, 5, 3, 6, 7]
k = 3
print(f"nums = {nums}, k = {k}")
print(f"window max = {sliding_window_max(nums, k)}")

In [ ]:
def sliding_window_max_verbose(nums, k):
    q = deque()
    result = []

    for i, num in enumerate(nums):
        print(f"Step {i}: value = {num}")

        while q and q[0] <= i - k:
            removed = q.popleft()
            print(f"  Remove expired {removed}")

        while q and nums[q[-1]] <= num:
            removed = q.pop()
            print(f"  Remove smaller {removed}")

        q.append(i)
        print(f"  Deque: {list(q)}")

        if i >= k - 1:
            result.append(nums[q[0]])
            print(f"  Window max = {nums[q[0]]}")
        print()

    return result

sliding_window_max_verbose([1, 3, -1, -3, 5, 3, 6, 7], 3)

---

## 9. LeetCode Problems

### LeetCode 496: Next Greater Element I

In [ ]:
def nextGreaterElement(nums1, nums2):
    next_greater = {}
    stack = []
    
    for num in nums2:
        while stack and stack[-1] < num:
            next_greater[stack.pop()] = num
        stack.append(num)
    
    return [next_greater.get(num, -1) for num in nums1]

print(nextGreaterElement([4,1,2], [1,3,4,2]))

### LeetCode 503: Next Greater Element II (Circular)

In [ ]:
def nextGreaterElements(nums):
    n = len(nums)
    result = [-1] * n
    stack = []
    
    for i in range(2 * n):
        idx = i % n
        while stack and nums[stack[-1]] < nums[idx]:
            result[stack.pop()] = nums[idx]
        if i < n:
            stack.append(idx)
    
    return result

print(nextGreaterElements([1,2,1]))

### LeetCode 739: Daily Temperatures

In [ ]:
def dailyTemperatures(temperatures):
    answer = [0] * len(temperatures)
    stack = []

    for i, temp in enumerate(temperatures):
        while stack and temperatures[stack[-1]] < temp:
            prev = stack.pop()
            answer[prev] = i - prev
        stack.append(i)

    return answer

print(dailyTemperatures([73,74,75,71,69,72,76,73]))

### LeetCode 84: Largest Rectangle in Histogram

In [ ]:
def largestRectangleArea(heights):
    stack = []
    max_area = 0
    extended = heights + [0]

    for i, h in enumerate(extended):
        while stack and extended[stack[-1]] > h:
            height = extended[stack.pop()]
            left = stack[-1] if stack else -1
            width = i - left - 1
            max_area = max(max_area, height * width)
        stack.append(i)

    return max_area

print(largestRectangleArea([2,1,5,6,2,3]))

### LeetCode 239: Sliding Window Maximum

In [ ]:
def maxSlidingWindow(nums, k):
    from collections import deque
    q = deque()
    result = []

    for i, num in enumerate(nums):
        while q and q[0] <= i - k:
            q.popleft()
        while q and nums[q[-1]] <= num:
            q.pop()
        q.append(i)
        if i >= k - 1:
            result.append(nums[q[0]])

    return result

print(maxSlidingWindow([1,3,-1,-3,5,3,6,7], 3))

### LeetCode 901: Online Stock Span

In [ ]:
class StockSpanner:
    def __init__(self):
        self.stack = []

    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span

spanner = StockSpanner()
for price in [100, 80, 60, 70, 60, 75, 85]:
    print(f"Price {price}: span = {spanner.next(price)}")

---

## 10. Pattern Recognition: When to Use What

### The Decision Process

**Ask yourself these questions:**

1. **Am I looking for the next/previous element that is greater/smaller?**
   → Use **Monotonic Stack**

2. **Am I finding max/min in every sliding window?**
   → Use **Monotonic Queue**

3. **Do I need the NEXT GREATER element?**
   → Use **Decreasing Stack** (pop when current > top)

4. **Do I need the NEXT SMALLER element?**
   → Use **Increasing Stack** (pop when current < top)

### Visual Decision Tree

```
                    What's the problem?
                           |
        +-----------------+-----------------+
        |                                   |
   Next/Previous                    Sliding Window
   element queries                    Max/Min
        |                                   |
        v                                   v
  Monotonic Stack                   Monotonic Queue
        |                                   |
        |                                   |
   Which one?                          Which one?
        |                                   |
   +----+----+                         +----+----+
   |         |                         |         |
  Next     Next                      Window   Window
 Greater  Smaller                      Max      Min
   |         |                         |         |
   v         v                         v         v
Decreasing Increasing              Decreasing Increasing
  Stack     Stack                     Deque     Deque
```

### Keyword Recognition

| Keywords in Problem | Use This |
|---------------------|----------|
| "next greater element" | Monotonic decreasing stack |
| "next smaller element" | Monotonic increasing stack |
| "previous greater" | Monotonic decreasing stack |
| "previous smaller" | Monotonic increasing stack |
| "sliding window maximum" | Monotonic decreasing deque |
| "sliding window minimum" | Monotonic increasing deque |
| "largest rectangle" | Monotonic increasing stack |
| "stock span" | Monotonic decreasing stack |
| "daily temperatures" | Monotonic decreasing stack |
| "trapping rain water" | Monotonic stack (boundaries) |

### Quick Reference Table

| Problem Type | Data Structure | Order | Pop When |
|--------------|----------------|-------|----------|
| Next Greater | Stack | Decreasing | current > top |
| Next Smaller | Stack | Increasing | current < top |
| Previous Greater | Stack | Decreasing | current ≥ top |
| Previous Smaller | Stack | Increasing | current ≤ top |
| Window Max | Deque | Decreasing | current ≥ back |
| Window Min | Deque | Increasing | current ≤ back |

### Common Patterns

1. **Distance-based answers** (like "days until warmer")
   - Store indices, not values
   - Answer = current_index - popped_index

2. **Value-based answers** (like "what is the next greater value")
   - Store indices
   - Answer = nums[current_index]

3. **Range/boundary problems** (like "how far can this extend")
   - Store indices
   - Use stack to find left and right boundaries

---

## 11. Common Mistakes and How to Avoid Them

### Mistake 1: Storing Values Instead of Indices

**Wrong**:
```python
stack.append(num)  # Storing the value
```

**Right**:
```python
stack.append(i)    # Storing the index
```

**Why indices?**
- You can always get the value: `nums[index]`
- You need indices for distances: `current_i - prev_i`
- You need indices for window membership checks
- You need indices for range calculations

### Mistake 2: Wrong Comparison Operator

**For Next Greater** (use `<`):
```python
while stack and nums[stack[-1]] < num:  # Correct
    # Pop smaller elements
```

**For Next Smaller** (use `>`):
```python
while stack and nums[stack[-1]] > num:  # Correct
    # Pop larger elements
```

**Common error**: Using `<=` or `>=` when you need `<` or `>`
- Use `<` or `>` for **strictly** greater/smaller
- Use `<=` or `>=` when equal values should also be popped

### Mistake 3: Forgetting to Remove Expired Indices in Sliding Window

**Wrong**:
```python
# Only removing smaller elements
while q and nums[q[-1]] <= num:
    q.pop()
```

**Right**:
```python
# FIRST remove expired, THEN remove smaller
while q and q[0] <= i - k:  # Remove expired
    q.popleft()
while q and nums[q[-1]] <= num:  # Remove smaller
    q.pop()
```

**Order matters!** Always remove expired indices before processing the current element.

### Mistake 4: Not Adding Sentinel in Histogram Problems

**Wrong**:
```python
for i, h in enumerate(heights):
    # Some bars never get popped
```

**Right**:
```python
extended = heights + [0]  # Add sentinel
for i, h in enumerate(extended):
    # The 0 forces all bars to pop
```

The sentinel (usually 0 for histogram) ensures all elements get processed.

### Mistake 5: Forgetting Remaining Stack Elements Have No Answer

**Wrong**:
```python
result = [0] * len(nums)  # Default 0
# Elements in stack never get updated
```

**Right**:
```python
result = [-1] * len(nums)  # Default -1
# Elements still in stack already have -1
```

Initialize with the "no answer" value so elements that never get popped already have the correct answer.

### Mistake 6: Confusing Decreasing vs Increasing

**Memory trick**:
- **Next GREATER** → **Decreasing** stack (we pop SMALLER elements)
- **Next SMALLER** → **Increasing** stack (we pop LARGER elements)

It's opposite of what you might expect! The stack order is opposite of what we're looking for.

---

## 12. Summary: The Big Picture

### What Makes Monotonic Structures Special?

**The core principle**: Remove useless candidates immediately instead of checking them repeatedly.

**Real-world analogy**: 
- Brute force = asking everyone in line individually
- Monotonic = keeping a smart waiting list that auto-updates

### Core Principles

1. **Monotonic Stack**: Maintains elements in increasing or decreasing order
   - Used for next/previous element queries
   - Each element pushed once, popped at most once
   - O(n) time complexity

2. **Monotonic Queue**: Deque with monotonic order
   - Used for sliding window max/min
   - Front always has the answer (max or min)
   - O(n) time for n windows

3. **Store Indices**: Almost always store indices, not values
   - Can access values anytime: `nums[index]`
   - Need indices for distances, ranges, windows

4. **Amortized O(n)**: Total operations bounded by 2n
   - n pushes maximum
   - n pops maximum
   - No element processed multiple times

### Quick Decision Guide

**Problem asks for next/previous element?**
→ Use **Monotonic Stack**

**Problem asks for window max/min?**
→ Use **Monotonic Queue**

**Looking for GREATER element?**
→ Use **Decreasing** order (pop smaller)

**Looking for SMALLER element?**
→ Use **Increasing** order (pop larger)

### The Mental Model

Think of the stack/queue as a **waiting list**:
- Elements join the waiting list
- When their answer arrives, they leave
- Useless elements are removed immediately
- The list stays organized (monotonic)

### Key Takeaways

| Use Case | Structure | Order | Example |
|----------|-----------|-------|---------|
| Next greater | Stack | Decreasing | Daily Temperatures |
| Next smaller | Stack | Increasing | Largest Rectangle |
| Previous greater | Stack | Decreasing | Stock Span |
| Previous smaller | Stack | Increasing | - |
| Window max | Deque | Decreasing | Sliding Window Max |
| Window min | Deque | Increasing | - |

### The Transformation

**From**: O(n²) brute force with nested loops and repeated comparisons

**To**: O(n) optimal solution with smart bookkeeping

**How**: By recognizing that once an element becomes useless, we can forget it forever.

### Final Thought

Monotonic structures are one of the most elegant optimization techniques in computer science. They transform seemingly complex problems into simple, linear-time solutions.

The key is recognizing the pattern: **"For each element, find the next/previous element that is greater/smaller"** or **"Find max/min in every window."**

Once you see these patterns, you know exactly what to do!